# MUTCD Multimodal RAG (v3) — Colab / HPRC

Hierarchical hybrid retrieval (dense + sparse + knowledge-graph proximity +
rule-type weighting) over the MUTCD 11th Edition, with a VLM answer step
that can run either **locally on GPU** or via an **API** (Qwen3-VL-32B by
default through DashScope).

Run cells top to bottom, once per session.

## 0. First-cell setup (Colab) — skip this entire section on HPRC

In [ ]:
# COLAB ONLY — comment out this whole cell if running on HPRC.
import sys, os

# Mount Google Drive — this is where the PDF, figures, Qdrant snapshot, and
# HF model cache persist across sessions.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# CRITICAL: tell every subprocess (including ingest_v3.py launched via !python)
# that this is a Colab environment. sys.modules-based detection does NOT cross
# process boundaries, but environment variables DO.
os.environ["MRAG_ENV"] = "colab"

# Clone or update the repo
if not os.path.isdir("/content/MRAG"):
    # v4: this is the stp2 repo now (directory stays /content/MRAG so no other
    # cell needs to change)
    !git clone https://github.com/hannanazad/MRAG_stp2.git /content/MRAG
else:
    !cd /content/MRAG && git pull

sys.path.insert(0, "/content/MRAG")

# ───────────────────────────────────────────────────────────────────────────
# Step 1 — torch + torchvision matched to Colab's CUDA 12.4.
# Colab's base image ships torch 2.5.1, but transformers >= 4.51 includes a
# torch.load CVE check (CVE-2025-32434) that refuses to load .bin checkpoints
# on torch < 2.6. BGE-M3 (and most HF models that ship .bin) need torch 2.6+
# in order for sentence-transformers to load them.
!pip install -q --index-url https://download.pytorch.org/whl/cu124 \
    torch==2.6.0 torchvision==0.21.0

# Step 2 — the rest of the dependencies from requirements.txt
!pip install -q -r /content/MRAG/requirements.txt

# Step 3 — pip's bulk -r resolver silently skips enforcing CEILINGS when a
# satisfying lower-bound version is already installed. Colab's base image
# ships newer transformers / huggingface_hub / tokenizers / torchao than we
# can use, so we force them to the right versions explicitly:
#   - transformers   <4.55   avoids PreTrainedModel circular-import regression
#   - huggingface_hub <0.35  avoids the API-removal regression that broke
#                            transformers 4.54.1's runtime version check
#   - tokenizers     <0.22   matched to transformers 4.54.x
#   - torchao        <0.14   0.13 is the last release supporting torch 2.6
#                            (0.14+ uses torch.int1, which is torch 2.7+ only)
!pip install -q --no-deps --force-reinstall \
    "transformers>=4.49,<4.55" \
    "huggingface_hub>=0.34,<0.35" \
    "tokenizers>=0.21,<0.22" \
    "torchao>=0.13,<0.14"

# Step 4 — sanity-check the critical import path before moving on.
!python -c "from transformers import PreTrainedModel; print('transformers import OK:', PreTrainedModel.__name__)"


In [ ]:
# Load ALL provider API keys from Colab Secrets (key icon, left sidebar).
# Missing ones are skipped — you only need keys for providers you'll use.
# NOTE: keys authenticate; they do NOT select the model. Select with
# CFG.set_vlm_model("fable" | "gemini-pro" | "frontier_qwen" | ...).
from google.colab import userdata
import os

def _load(env_name, *secret_names):
    for s in secret_names:
        try:
            os.environ[env_name] = userdata.get(s)
            print(f"{env_name}: loaded (from secret {s!r})")
            return
        except Exception:
            continue
    print(f"{env_name}: no matching secret (skipped)")

_load("DASHSCOPE_API_KEY", "DASHSCOPE_API_KEY", "QWEN")   # "QWEN" = your existing secret name
_load("ANTHROPIC_API_KEY", "ANTHROPIC_API_KEY")
_load("GEMINI_API_KEY",    "GEMINI_API_KEY")


## 1. Environment sanity check (both Colab and HPRC)

In [ ]:
import os, sys
sys.path.insert(0, "/content/MRAG" if os.path.isdir("/content/MRAG") else ".")

from mrag.config import CFG

print("Environment :", CFG.environment)
print("Base dir    :", CFG.base_dir, "| exists:", CFG.base_dir.exists())
print("PDF path    :", CFG.pdf_path, "| exists:", CFG.pdf_path.exists())
print("Qdrant dir  :", CFG.qdrant_dir)
print("Cache dir   :", CFG.cache_dir)
print("HF cache    :", CFG.hf_home)
print("VLM provider:", CFG.vlm_provider)
print("VLM model   :", CFG.vlm_model_api if CFG.vlm_provider == "api" else CFG.vlm_model)
print("API key set :", bool(os.environ.get(CFG.api_key_env_var)))

assert CFG.pdf_path.exists(), (
    f"No PDF found at {CFG.pdf_path} or anywhere in {CFG.base_dir}. "
    f"Upload the MUTCD PDF to {CFG.base_dir} before continuing."
)


## 1.5 Recover or build the Qdrant vector store

If you've run ingestion successfully before, this restores the existing
Drive snapshot into the fast local Colab disk in seconds. If no snapshot
exists yet, it falls through to running full ingestion (first time only,
~30–45 min; subsequent reruns are much faster since chunks/figures/KG are
cached and only re-embedding + upsert happens, ~5 min).

In [ ]:
import shutil, tarfile, json
from pathlib import Path
from qdrant_client import QdrantClient

local_qdrant = CFG.qdrant_dir                      # /content/qdrant_db (fast local SSD)
drive_qdrant_tar = CFG.base_dir / "qdrant_db.tar"  # snapshot on Drive

def _qdrant_ok(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        c = QdrantClient(path=str(path))
        try:
            names = {col.name for col in c.get_collections().collections}
            return CFG.coll_chunks in names
        finally:
            c.close()
    except Exception:
        return False

def _v4_done() -> bool:
    """True once the v4 (caption-below) figure extraction has run — the
    signal that crops, KG, and figure embeddings are the corrected ones."""
    try:
        with open(CFG.figures_jsonl) as f:
            first = json.loads(next(f))
        return str(first.get("extraction_method", "")).startswith("caption_below_v2")
    except Exception:
        return False

if _qdrant_ok(local_qdrant) and _v4_done():
    print(f"Qdrant populated at {local_qdrant} and figures are v4. Nothing to do.")

elif drive_qdrant_tar.exists() and _v4_done():
    # NOTE: this assumes the tar was snapshotted AFTER the first v4 ingest
    # (cell 1.6). If it predates v4, delete Drive/MRAG/qdrant_db.tar once and
    # re-run this cell.
    print(f"Restoring v4 Qdrant snapshot from {drive_qdrant_tar} "
          f"({drive_qdrant_tar.stat().st_size/1e6:.0f} MB)...")
    shutil.rmtree(local_qdrant, ignore_errors=True)
    local_qdrant.mkdir(parents=True, exist_ok=True)
    with tarfile.open(drive_qdrant_tar, "r") as tar:
        tar.extractall(local_qdrant.parent)
    assert _qdrant_ok(local_qdrant), "Extraction completed but collection not found."
    print("Local Qdrant verified OK.")

else:
    if drive_qdrant_tar.exists() and not _v4_done():
        print("A Qdrant snapshot exists on Drive but figures are still v1 —")
        print("ignoring the stale snapshot and running the v4 ingest.")
        print("After it finishes, run cell 1.6 to overwrite the snapshot.")
    else:
        print("No usable v4 store found. Running ingestion (v4).")
    print("First v4 run re-extracts all figures, rebuilds the KG, and")
    print("re-embeds figure crops: expect ~30-45 min on an A100.")
    !cd /content/MRAG && python scripts/ingest_v4.py
    assert _qdrant_ok(local_qdrant), "Ingestion finished but Qdrant still empty — check the log."
    assert _v4_done(), "Ingestion ran but figures.jsonl is not v2 — check the log."
    print("v4 ingestion verified OK.")


## 1.6 Snapshot Qdrant back to Drive (Colab only)

Run this after any successful ingestion so the *next* session can restore
instantly via the recovery cell above, instead of re-running ingestion.

In [ ]:
if CFG.environment == "colab":
    local_qdrant = CFG.qdrant_dir  # define here so this cell can run independently
    drive_qdrant_tar = CFG.base_dir / "qdrant_db.tar"
    
    if local_qdrant.exists():
        print(f"Snapshotting {local_qdrant} -> {drive_qdrant_tar} ...")
        drive_qdrant_tar.parent.mkdir(parents=True, exist_ok=True)
        with tarfile.open(drive_qdrant_tar, "w") as tar:
            tar.add(local_qdrant, arcname=local_qdrant.name)
        print(f"Snapshot complete ({drive_qdrant_tar.stat().st_size/1e6:.0f} MB).")
    else:
        print("No local Qdrant to snapshot.")


## 2. Initialize the pipeline (one time per session, ~30–60 s in API mode)

In [ ]:
import logging, os
logging.basicConfig(level=logging.INFO, format='%(name)s - %(message)s')

# v4.6 guard: warn (don't block) — clients are built lazily per provider.
# Select your model BEFORE or AFTER init, e.g. CFG.set_vlm_model("fable").
if CFG.vlm_provider == "api" and not os.environ.get(CFG.api_key_env_var):
    print(f"NOTE: no key in {CFG.api_key_env_var!r} for the current model "
          f"({CFG.vlm_model_api!r}). Init will proceed; load that key or "
          f"switch models with CFG.set_vlm_model(...) before asking.")

from mrag.ask import init_pipeline
pipeline = init_pipeline()
print("VLM loaded :", pipeline.vlm.loaded_name if pipeline.vlm else "none")
print("KG         :", pipeline.kg.g.number_of_nodes(), "nodes,", pipeline.kg.g.number_of_edges(), "edges")


## 3. Ask

In [ ]:
from mrag.ask import ask

_ = ask("What is required when installing a STOP sign at an all-way stop intersection?")


In [ ]:
_ = ask("Explain Figure 2B-1 and the plaques it shows", show_scores=True)


In [ ]:
_ = ask("What does MUTCD say about pedestrian hybrid beacons?", show_text=True)


# v4 router smoke test: a definitional question should retrieve NO figures.
_ = ask("What does 'shall' mean in the MUTCD?", show_scores=True)
# check the printed debug: figure_router.needs_figures should be False


## 4. Inspect the knowledge graph

In [ ]:
kg = pipeline.kg
print(kg.g.number_of_nodes(), "nodes,", kg.g.number_of_edges(), "edges")


In [ ]:
# Neighbourhood of a sign code
# (fill in a real sign code node id from your graph, e.g. "signcode:R1-1")


In [ ]:
# Which figures does Section 2B.04 directly link to via cross-references in its chunks?


## 5. Debug retrieval (no VLM call)

In [ ]:
from mrag.retrieval import Retriever
